In [8]:
lower = 0.0   # cannot short-sell (negative allocation)
upper = 1.0   # cannot put more than 100% in one asset
# n_dims = number of assets

In [9]:
def portfolio_fitness(weights, returns, cov_matrix, risk_aversion=0.5):
    n = len(weights)
    weights = [max(0.05, w) for w in weights]  # minimum 5% per asset
    total = sum(weights)
    if total == 0:
        return float('inf')
    w = [weights[i] / total for i in range(n)]
    
    portfolio_return = sum(w[i] * returns[i] for i in range(n))
    
    portfolio_risk = 0
    for i in range(n):
        for j in range(n):
            portfolio_risk += w[i] * w[j] * cov_matrix[i][j]
    
    return risk_aversion * portfolio_risk - (1 - risk_aversion) * portfolio_return

In [10]:
import random
import math

def special_forces_algorithm(fitness_func, lower, upper,
                              n_dims=3,
                              population_size=30,
                              iterations=500,
                              tv1=0.5, tv2=0.3,
                              p0=0.25, k=0.1):

    soldiers = [
        [random.uniform(lower, upper) for _ in range(n_dims)]
        for _ in range(population_size)
    ]
    fitnesses = [fitness_func(s) for s in soldiers]

    best_idx = min(range(population_size), key=lambda i: fitnesses[i])
    global_best = soldiers[best_idx].copy()
    global_best_fitness = fitnesses[best_idx]

    for t in range(1, iterations + 1) :
        # Unmaned search
        
        c = k * (lower + (1 - t / iterations) * (upper - lower))
        uav_results = []
        for i in range(population_size):
            v = [random.gauss(0, 1) for _ in range(n_dims)]
            norm = math.sqrt(sum(vi**2 for vi in v))
            if norm == 0:
                norm = 1e-10
            v = [vi / norm * abs(c) for vi in v]
            xu = [soldiers[i][d] + v[d] for d in range(n_dims)]
            xu = [max(lower, min(upper, xu[d])) for d in range(n_dims)]
            uav_results.append(xu)

        for xu in uav_results:
            f_xu = fitness_func(xu)
            if f_xu < global_best_fitness:
                global_best = xu.copy()
                global_best_fitness = f_xu

        instruction = (1 - 0.15 * random.random()) * (1 - t / iterations)

        p_loss = p0 * math.cos(math.pi * t / (2 * iterations))

        w = 0.75 - 0.55 * math.atan((t / iterations) ** (2 * math.pi))

        x_ave = [
            sum(soldiers[i][d] for i in range(population_size)) / population_size
            for d in range(n_dims)
        ]

        new_soldiers = []

        for i in range(population_size):
            r1 = random.random()
            r2 = random.random()

            if instruction >= tv1:
                # PHASE 1: EXPLORATION

                if r1 >= 0.5:
                    # Large scale search
                    
                    sign = 1 if random.random() >= 0.5 else -1
                    new_pos = [
                        r1 * (global_best[d] - soldiers[i][d])
                        + sign * (1 - r1) * (upper - lower)
                        for d in range(n_dims)
                    ]
                else:
                    # Raid
                    
                    if random.random() < p_loss:
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]
                    denom = f_i + f_aim
                    if denom == 0:
                        denom = 1e-10
                    ratio = f_i / denom

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]

            elif tv2 < instruction < tv1:
                # PHASE 2: TRANSITION 

                if r2 >= 0.5:
                    # Raid 
                    if random.random() < p_loss:
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]

                    denom = f_i + f_aim
                    if denom == 0 or math.isnan(denom) or math.isinf(denom):
                        denom = 1e-10
                    ratio = f_i / denom
                    

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]
                else:
                    # move toward global best 
                    new_pos = [
                        instruction * (global_best[d] - soldiers[i][d]) + 0.1 * soldiers[i][d]
                        for d in range(n_dims)
                    ]

            else:
                # PHASE 3: EXPLOITATION
                # converge around global best using average position as reference
                r = [random.uniform(-1, 1) for _ in range(n_dims)]
                new_pos = [
                    global_best[d] + r[d] * abs(global_best[d] - x_ave[d])
                    for d in range(n_dims)
                ]

            new_pos = [max(lower, min(upper, new_pos[d])) for d in range(n_dims)]
            new_soldiers.append(new_pos)

        soldiers = new_soldiers
        fitnesses = [fitness_func(s) for s in soldiers]

        for i in range(population_size):
            if fitnesses[i] < global_best_fitness:
                global_best = soldiers[i].copy()
                global_best_fitness = fitnesses[i]

        # if t % 10 == 0:
        #     if instruction >= tv1:
        #         phase = "1-Exploration"
        #     elif instruction > tv2:
        #         phase = "2-Transition"
        #     else:
        #         phase = "3-Exploitation"
        #     print(f"Iter {t}: best={global_best_fitness:.4e}, instruction={instruction:.3f}, phase={phase}")

    return global_best, global_best_fitness

In [11]:
import yfinance as yf
import math

# download 5 years of daily prices for 5 assets
tickers = ['AAPL', 'GOOGL', 'MSFT', 'JPM', 'GLD']
data = yf.download(tickers, start='2019-01-01', end='2024-01-01')['Close']

# compute daily returns
daily_returns = []
for i in range(1, len(data)):
    row = []
    for ticker in tickers:
        prev = data[ticker].iloc[i-1]
        curr = data[ticker].iloc[i]
        row.append((curr - prev) / prev)
    daily_returns.append(row)

# expected annual return for each asset
n_days = len(daily_returns)
n_assets = len(tickers)

mean_returns = []
for j in range(n_assets):
    mean_r = sum(daily_returns[i][j] for i in range(n_days)) / n_days
    mean_returns.append(mean_r * 252)  # annualize (252 trading days)

# covariance matrix
cov_matrix = [[0.0]*n_assets for _ in range(n_assets)]
for i in range(n_assets):
    for j in range(n_assets):
        mean_i = mean_returns[i] / 252
        mean_j = mean_returns[j] / 252
        cov = sum(
            (daily_returns[d][i] - mean_i) * (daily_returns[d][j] - mean_j)
            for d in range(n_days)
        ) / (n_days - 1)
        cov_matrix[i][j] = cov * 252  # annualize

[*********************100%***********************]  5 of 5 completed


In [15]:
# exact same SFA code, just change the fitness function and bounds
pos, fit = special_forces_algorithm(
    fitness_func   = lambda w: portfolio_fitness(w, mean_returns, cov_matrix, risk_aversion=0.8),
    lower          = 0.0,
    upper          = 1.0,
    n_dims         = len(tickers),    # 5 assets = 5 dimensions
    population_size = 30,
    iterations     = 500
)

# normalize final weights
total = sum(pos)
final_weights = [pos[i] / total for i in range(len(pos))]

print("Optimal allocation:")
for i, ticker in enumerate(tickers):
    print(f"  {ticker}: {final_weights[i]*100:.1f}%")

Optimal allocation:
  AAPL: 32.6%
  GOOGL: 0.5%
  MSFT: 12.0%
  JPM: 4.1%
  GLD: 50.7%
